In [ ]:
!pip install -q "ultralytics>=8.3.0"

In [ ]:
# Core imports
import os
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch

from ultralytics import YOLO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!ls /kaggle/input/datasets/khadijamellak

In [ ]:
!ls /kaggle/input/datasets/khadijamellak/ena24-dataset

In [ ]:
# après redémarrage
import torch, platform

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
from pathlib import Path
import shutil

# Dataset Kaggle (déjà extrait)
KAGGLE_DATASET_PATH = Path("/kaggle/input/datasets/khadijamellak/ena24-dataset")

# Dossier de travail (écriture autorisée)
EXTRACT_ROOT = Path("/kaggle/working/data")

# Nettoyage
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

# Vérification
if not KAGGLE_DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset introuvable : {KAGGLE_DATASET_PATH}")

# Copie complète du dataset
shutil.copytree(KAGGLE_DATASET_PATH, EXTRACT_ROOT, dirs_exist_ok=True)

print("✅ Dataset copié dans :", EXTRACT_ROOT)

# Affichage contenu
for p in EXTRACT_ROOT.iterdir():
    print(p.name, "(DIR)" if p.is_dir() else "(FILE)")

In [ ]:
from pathlib import Path

# Chemin racine du dataset
KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/khadijamellak/ena24-dataset")

# Sous-dossiers
KAGGLE_IMAGES_DIR = KAGGLE_DATASET_DIR / "images"
KAGGLE_METADATA_DIR = KAGGLE_DATASET_DIR / "metadata"
KAGGLE_CKPT_DIR = KAGGLE_DATASET_DIR / "checkpoints"

# Trouver automatiquement un JSON dans metadata
json_files = list(KAGGLE_METADATA_DIR.glob("*.json"))
KAGGLE_JSON_PATH = json_files[0] if json_files else None

In [ ]:
from pathlib import Path

# Vérifications de base
if not KAGGLE_DATASET_DIR.exists():
    raise FileNotFoundError(f"❌ Dataset introuvable: {KAGGLE_DATASET_DIR}")

if not KAGGLE_IMAGES_DIR.exists():
    raise FileNotFoundError(f"❌ Dossier images introuvable: {KAGGLE_IMAGES_DIR}")

if not KAGGLE_JSON_PATH.exists():
    raise FileNotFoundError(f"❌ JSON introuvable: {KAGGLE_JSON_PATH}")

print("✅ Dataset trouvé :", KAGGLE_DATASET_DIR)
print("✅ Images trouvées :", KAGGLE_IMAGES_DIR)
print("✅ JSON trouvé :", KAGGLE_JSON_PATH)

if KAGGLE_CKPT_DIR is not None:
    if KAGGLE_CKPT_DIR.exists():
        print("✅ Checkpoint dir trouvé :", KAGGLE_CKPT_DIR)
        ckpts = list(KAGGLE_CKPT_DIR.rglob("*.pt"))
        if ckpts:
            print("Checkpoint files found:")
            for p in ckpts:
                print(" -", p)
        else:
            print("⚠️ Aucun fichier .pt trouvé dans", KAGGLE_CKPT_DIR)
    else:
        print("⚠️ Dossier checkpoints introuvable :", KAGGLE_CKPT_DIR)

print("✅ Entrées Kaggle valides")

In [ ]:
import shutil
from pathlib import Path



WORK_ROOT = Path("/kaggle/working/data")
LOCAL_IMAGES = WORK_ROOT / "images"
LOCAL_JSON = WORK_ROOT / "ena24.json"

# Vérifications
assert KAGGLE_IMAGES_DIR.exists(), f"❌ Images source introuvables: {KAGGLE_IMAGES_DIR}"
assert KAGGLE_JSON_PATH.exists(), f"❌ JSON source introuvable: {KAGGLE_JSON_PATH}"

# Nettoyage ancien dossier
if LOCAL_IMAGES.exists():
    shutil.rmtree(LOCAL_IMAGES)

# Copie images
shutil.copytree(KAGGLE_IMAGES_DIR, LOCAL_IMAGES)

# Copie JSON
shutil.copy2(KAGGLE_JSON_PATH, LOCAL_JSON)

# Stats
nb_jpg = sum(1 for _ in LOCAL_IMAGES.rglob("*.jpg"))
nb_png = sum(1 for _ in LOCAL_IMAGES.rglob("*.png"))

print("✅ Images copiées vers:", LOCAL_IMAGES)
print("✅ JSON copié vers   :", LOCAL_JSON)
print(f"📊 Images: {nb_jpg} jpg | {nb_png} png")

In [ ]:
# ============================================================
# 1) Install EfficientDet stack
# ============================================================
!pip install -q effdet pycocotools

In [ ]:
# ============================================================
# 2) Resolve dataset path robustly on Kaggle
# ============================================================
from pathlib import Path
import json
import random
import shutil
import os

POSSIBLE_ROOTS = [
    Path("/kaggle/input/ena24"),
    Path("/kaggle/input/ena24-dataset"),
    Path("/kaggle/input/datasets/khadijamellak/ena24-dataset"),  # fallback for your current draft
]

DATASET_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists():
        DATASET_ROOT = p
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Dataset not found. Add your Kaggle dataset in the notebook sidebar, "
        "then check its exact mounted path under /kaggle/input/..."
    )

IMAGES_DIR = DATASET_ROOT / "images"
METADATA_DIR = DATASET_ROOT / "metadata"

json_files = list(METADATA_DIR.glob("*.json"))
if len(json_files) == 0:
    raise FileNotFoundError(f"No JSON file found in {METADATA_DIR}")

ANNOTATIONS_JSON = json_files[0]

print("DATASET_ROOT   :", DATASET_ROOT)
print("IMAGES_DIR     :", IMAGES_DIR)
print("ANNOTATIONS_JSON:", ANNOTATIONS_JSON)

In [ ]:
# ============================================================
# 3) Load COCO-style annotations
# ============================================================
with open(ANNOTATIONS_JSON, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

print("Top-level keys:", list(coco_data.keys()))
print("Number of images      :", len(coco_data.get("images", [])))
print("Number of annotations :", len(coco_data.get("annotations", [])))
print("Number of categories  :", len(coco_data.get("categories", [])))

assert "images" in coco_data and "annotations" in coco_data and "categories" in coco_data, \
    "Expected a COCO-style JSON with images / annotations / categories."

display(coco_data["categories"][:10])

In [ ]:
# ============================================================
# 4) Build train/val split at image level
#    - keeps only images that have at least 1 annotation
# ============================================================
from collections import defaultdict

SEED = 42
VAL_RATIO = 0.2

random.seed(SEED)

image_id_to_info = {img["id"]: img for img in coco_data["images"]}

anns_per_image = defaultdict(list)
for ann in coco_data["annotations"]:
    if ann.get("iscrowd", 0) == 1:
        continue
    if ann.get("bbox") is None:
        continue
    if ann["image_id"] in image_id_to_info:
        anns_per_image[ann["image_id"]].append(ann)

valid_image_ids = [img_id for img_id, anns in anns_per_image.items() if len(anns) > 0]
random.shuffle(valid_image_ids)

n_val = max(1, int(len(valid_image_ids) * VAL_RATIO))
val_image_ids = set(valid_image_ids[:n_val])
train_image_ids = set(valid_image_ids[n_val:])

print("Train images:", len(train_image_ids))
print("Val images  :", len(val_image_ids))

WORK_DIR = Path("/kaggle/working/effdet_ena24")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSON = WORK_DIR / "train_coco.json"
VAL_JSON = WORK_DIR / "val_coco.json"

def make_subset_json(selected_image_ids, out_path):
    selected_images = [image_id_to_info[i] for i in selected_image_ids]
    selected_annotations = [
        ann for ann in coco_data["annotations"]
        if ann["image_id"] in selected_image_ids and ann.get("iscrowd", 0) == 0
    ]
    subset = {
        "images": selected_images,
        "annotations": selected_annotations,
        "categories": coco_data["categories"]
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(subset, f)

make_subset_json(train_image_ids, TRAIN_JSON)
make_subset_json(val_image_ids, VAL_JSON)

print("Saved:", TRAIN_JSON)
print("Saved:", VAL_JSON)

In [ ]:
# ============================================================
# 5) Category remapping to contiguous labels [1..num_classes]
#    EfficientDet expects class ids to be contiguous.
# ============================================================
categories = coco_data["categories"]
orig_cat_ids = [c["id"] for c in categories]
cat_id_to_label = {cat_id: i + 1 for i, cat_id in enumerate(orig_cat_ids)}   # reserve 0 as background
label_to_name = {i + 1: c["name"] for i, c in enumerate(categories)}

NUM_CLASSES = len(categories)
print("NUM_CLASSES:", NUM_CLASSES)
print(label_to_name)

In [ ]:
# ============================================================
# Categories / label mapping
# Exclude class 8 = Human, class 9 = Vehicle
# ============================================================
EXCLUDED_CLASSES = {8, 9}

filtered_categories = [
    c for c in coco_data["categories"]
    if c["id"] not in EXCLUDED_CLASSES
]

filtered_cat_ids = [c["id"] for c in filtered_categories]

cat_id_to_label = {cat_id: i + 1 for i, cat_id in enumerate(filtered_cat_ids)}
label_to_name = {i + 1: c["name"] for i, c in enumerate(filtered_categories)}

NUM_CLASSES = len(filtered_categories)

print("Excluded classes:", EXCLUDED_CLASSES)
print("Remaining categories:", [c["name"] for c in filtered_categories])
print("NUM_CLASSES:", NUM_CLASSES)
print("cat_id_to_label:", cat_id_to_label)

In [ ]:
# ============================================================
# Dataset + DataLoader for EfficientDet
# - skips missing image files
# - removes excluded classes 8 and 9
# - keeps only valid boxes
# ============================================================
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pycocotools.coco import COCO
from pathlib import Path

IMG_SIZE = 512
MAX_DETECTIONS = 100
BATCH_SIZE = 4
NUM_WORKERS = 2
EXCLUDED_CLASSES = {8, 9}

class CocoEffDetDataset(Dataset):
    def __init__(self, images_dir, ann_json, cat_id_to_label, img_size=512, excluded_classes=None):
        self.images_dir = Path(images_dir)
        self.coco = COCO(str(ann_json))
        self.cat_id_to_label = cat_id_to_label
        self.img_size = img_size
        self.excluded_classes = excluded_classes if excluded_classes is not None else set()

        all_image_ids = list(self.coco.imgs.keys())
        filtered = []
        missing_files = []
        empty_after_filter = []

        for img_id in all_image_ids:
            img_info = self.coco.loadImgs([img_id])[0]
            img_path = self.images_dir / img_info["file_name"]

            if not img_path.exists():
                missing_files.append(img_info["file_name"])
                continue

            ann_ids = self.coco.getAnnIds(imgIds=[img_id])
            anns = self.coco.loadAnns(ann_ids)

            valid = []
            for a in anns:
                if a.get("iscrowd", 0) == 1:
                    continue
                if a.get("bbox") is None:
                    continue
                if a["category_id"] in self.excluded_classes:
                    continue
                if a["category_id"] not in self.cat_id_to_label:
                    continue

                x, y, w, h = a["bbox"]
                if w <= 1 or h <= 1:
                    continue

                valid.append(a)

            if len(valid) > 0:
                filtered.append(img_id)
            else:
                empty_after_filter.append(img_info["file_name"])

        self.image_ids = filtered
        self.missing_files = missing_files
        self.empty_after_filter = empty_after_filter

        print(f"Valid images kept: {len(self.image_ids)}")
        print(f"Missing image files skipped: {len(self.missing_files)}")
        print(f"Images with no valid annotations after filtering skipped: {len(self.empty_after_filter)}")

        if len(self.missing_files) > 0:
            print("First missing files:", self.missing_files[:10])

        if len(self.empty_after_filter) > 0:
            print("First empty-after-filter files:", self.empty_after_filter[:10])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = self.images_dir / img_info["file_name"]

        image = Image.open(img_path).convert("RGB")

        orig_w, orig_h = image.size
        image = image.resize((self.img_size, self.img_size))
        image = np.array(image).astype(np.float32) / 255.0
        image = torch.tensor(image).permute(2, 0, 1)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []

        sx = self.img_size / orig_w
        sy = self.img_size / orig_h

        for ann in anns:
            if ann.get("iscrowd", 0) == 1:
                continue
            if ann.get("bbox") is None:
                continue
            if ann["category_id"] in self.excluded_classes:
                continue
            if ann["category_id"] not in self.cat_id_to_label:
                continue

            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue

            x1 = x * sx
            y1 = y * sy
            x2 = (x + w) * sx
            y2 = (y + h) * sy

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_label[ann["category_id"]])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "bbox": boxes,
            "cls": labels,
            "img_size": torch.tensor([self.img_size, self.img_size], dtype=torch.float32),
            "img_scale": torch.tensor([1.0], dtype=torch.float32),
            "image_id": img_id,
        }

        return image, target


def collate_fn(batch):
    images = []
    bboxes = []
    classes = []
    img_sizes = []
    img_scales = []
    image_ids = []

    max_instances = min(
        MAX_DETECTIONS,
        max(item[1]["bbox"].shape[0] for item in batch)
    )

    for image, target in batch:
        images.append(image)

        n = min(target["bbox"].shape[0], max_instances)

        bbox_pad = torch.zeros((max_instances, 4), dtype=torch.float32)
        cls_pad = torch.full((max_instances,), -1, dtype=torch.int64)

        if n > 0:
            bbox_pad[:n] = target["bbox"][:n]
            cls_pad[:n] = target["cls"][:n]

        bboxes.append(bbox_pad)
        classes.append(cls_pad)
        img_sizes.append(target["img_size"])
        img_scales.append(target["img_scale"])
        image_ids.append(target["image_id"])

    images = torch.stack(images)
    targets = {
        "bbox": torch.stack(bboxes),
        "cls": torch.stack(classes),
        "img_size": torch.stack(img_sizes),
        "img_scale": torch.stack(img_scales),
        "image_id": image_ids,
    }
    return images, targets


train_ds = CocoEffDetDataset(
    IMAGES_DIR,
    TRAIN_JSON,
    cat_id_to_label,
    img_size=IMG_SIZE,
    excluded_classes=EXCLUDED_CLASSES,
)

val_ds = CocoEffDetDataset(
    IMAGES_DIR,
    VAL_JSON,
    cat_id_to_label,
    img_size=IMG_SIZE,
    excluded_classes=EXCLUDED_CLASSES,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

In [ ]:
# ============================================================
# 7) Build EfficientDet model CORRECTLY for training
# ============================================================
import torch
from effdet import get_efficientdet_config, EfficientDet, DetBenchTrain
from effdet.efficientdet import HeadNet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "tf_efficientdet_d0"

config = get_efficientdet_config(MODEL_NAME)
config.num_classes = NUM_CLASSES
config.image_size = (IMG_SIZE, IMG_SIZE)

net = EfficientDet(config, pretrained_backbone=True)
net.class_net = HeadNet(config, num_outputs=config.num_classes)

# IMPORTANT: create_labeler=True enables internal anchor target generation
model = DetBenchTrain(net, config)
model = model.to(DEVICE)

print("Device:", DEVICE)
print("Model :", MODEL_NAME)
print("Image size:", config.image_size)
print("Num classes:", config.num_classes)

In [ ]:
# ============================================================
# 8) Optimizer / scheduler
# ============================================================
import math

EPOCHS = 10
LR = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

BEST_MODEL_PATH = WORK_DIR / f"{MODEL_NAME}_ena24_best.pth"
print("Best model will be saved to:", BEST_MODEL_PATH)

In [ ]:
# ============================================================
# Robust checkpointing for Kaggle:
# - best model
# - last checkpoint every epoch
# - emergency checkpoint on interruption
# - optional persistent upload to a private Kaggle dataset
# ============================================================
import os
import json
import time
import signal
import shutil
import subprocess
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
CKPT_DIR = Path("/kaggle/working/effdet_ena24_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CKPT_DIR / "best_model.pth"
LAST_CKPT_PATH = CKPT_DIR / "last_checkpoint.pth"
EMERGENCY_CKPT_PATH = CKPT_DIR / "emergency_checkpoint.pth"
HISTORY_CSV_PATH = CKPT_DIR / "history.csv"
TRAIN_STATE_JSON = CKPT_DIR / "train_state.json"

print("Checkpoint directory:", CKPT_DIR)

# ------------------------------------------------------------
# Optional: automatic persistent backup to a private Kaggle dataset
# IMPORTANT:
# - Requires Internet ON in Kaggle notebook
# - Requires Kaggle API credentials available
# - Replace with your dataset id once created
# ------------------------------------------------------------
ENABLE_KAGGLE_DATASET_BACKUP = False
KAGGLE_DATASET_ID = "khadijamellak/ena24-effdet-checkpoints"  # <- change if needed
KAGGLE_DATASET_TITLE = "ena24-effdet-checkpoints"

def kaggle_api_is_available():
    return shutil.which("kaggle") is not None

def write_dataset_metadata(folder: Path, dataset_id: str, title: str):
    meta = {
        "title": title,
        "id": dataset_id,
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(folder / "dataset-metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

def create_or_update_kaggle_dataset(folder: Path, dataset_id: str, title: str, version_note: str):
    """
    Publish checkpoints to a Kaggle Dataset for true persistence.
    This will fail if:
    - Internet is OFF
    - Kaggle CLI/auth is unavailable
    - dataset_id is invalid
    """
    try:
        write_dataset_metadata(folder, dataset_id, title)

        if not kaggle_api_is_available():
            print("[WARN] Kaggle CLI not found. Skipping persistent upload.")
            return

        # Try version first; if it fails, try create
        cmd_version = [
            "kaggle", "datasets", "version",
            "-p", str(folder),
            "-m", version_note,
            "-r", "zip",
            "-q"
        ]
        result = subprocess.run(cmd_version, capture_output=True, text=True)

        if result.returncode == 0:
            print("[INFO] Persistent backup updated on Kaggle dataset.")
            return

        cmd_create = [
            "kaggle", "datasets", "create",
            "-p", str(folder),
            "-u",
            "-r", "zip",
            "-q"
        ]
        result2 = subprocess.run(cmd_create, capture_output=True, text=True)

        if result2.returncode == 0:
            print("[INFO] Persistent backup dataset created on Kaggle.")
        else:
            print("[WARN] Kaggle dataset backup failed.")
            print(result2.stderr[:1000])

    except Exception as e:
        print(f"[WARN] Persistent Kaggle backup exception: {e}")

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def save_history_csv(history, path):
    pd.DataFrame(history).to_csv(path, index=False)

def save_train_state(epoch, best_val_loss, history, interrupted=False):
    state = {
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "interrupted": interrupted,
        "num_records": len(history),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(TRAIN_STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

def build_checkpoint_dict(model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history):
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val_loss,
        "history": history,
        "model_name": MODEL_NAME,
        "img_size": IMG_SIZE,
        "num_classes": NUM_CLASSES,
        "label_to_name": label_to_name,
    }

def save_last_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history):
    ckpt = build_checkpoint_dict(
        model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history
    )
    torch.save(ckpt, LAST_CKPT_PATH)
    save_history_csv(history, HISTORY_CSV_PATH)
    save_train_state(epoch, best_val_loss, history, interrupted=False)

def save_best_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history):
    ckpt = build_checkpoint_dict(
        model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history
    )
    torch.save(ckpt, BEST_MODEL_PATH)

def save_emergency_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history):
    ckpt = build_checkpoint_dict(
        model, optimizer, scheduler, epoch, train_loss, val_loss, best_val_loss, history
    )
    torch.save(ckpt, EMERGENCY_CKPT_PATH)
    save_history_csv(history, HISTORY_CSV_PATH)
    save_train_state(epoch, best_val_loss, history, interrupted=True)

# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------
RESUME = True

start_epoch = 1
history = []
best_val_loss = float("inf")

if RESUME and LAST_CKPT_PATH.exists():
    print(f"[INFO] Resuming from {LAST_CKPT_PATH}")
    ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    history = ckpt.get("history", [])
    print(f"[INFO] Resume start_epoch={start_epoch}, best_val_loss={best_val_loss:.6f}")

# ------------------------------------------------------------
# Loss extraction
# ------------------------------------------------------------
def get_total_loss(output):
    if isinstance(output, dict):
        if "loss" in output:
            return output["loss"]
        if "total_loss" in output:
            return output["total_loss"]
    if torch.is_tensor(output):
        return output
    if isinstance(output, (list, tuple)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 0:
                return item
    raise RuntimeError(f"Could not extract loss from output type: {type(output)}")

# ------------------------------------------------------------
# Train / Val loops
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, total=len(loader))

    for images, targets in pbar:
        images = images.to(device)
        targets = {k: v.to(device) if torch.is_tensor(v) else v for k, v in targets.items()}

        optimizer.zero_grad()
        output = model(images, targets)
        loss = get_total_loss(output)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_description(f"train_loss={loss.item():.4f}")

    return running_loss / max(1, len(loader))

@torch.no_grad()
def valid_one_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0
    pbar = tqdm(loader, total=len(loader))

    for images, targets in pbar:
        images = images.to(device)
        targets = {k: v.to(device) if torch.is_tensor(v) else v for k, v in targets.items()}

        output = model(images, targets)
        loss = get_total_loss(output)

        running_loss += loss.item()
        pbar.set_description(f"val_loss={loss.item():.4f}")

    return running_loss / max(1, len(loader))

# ------------------------------------------------------------
# Interruption-safe training
# ------------------------------------------------------------
stop_requested = {"flag": False}

def _request_stop(signum, frame):
    print(f"\n[WARN] Received signal {signum}. Will save emergency checkpoint and stop.")
    stop_requested["flag"] = True

signal.signal(signal.SIGTERM, _request_stop)
signal.signal(signal.SIGINT, _request_stop)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
last_epoch_seen = start_epoch - 1
last_train_loss = None
last_val_loss = None

try:
    for epoch in range(start_epoch, EPOCHS + 1):
        last_epoch_seen = epoch
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
        val_loss = valid_one_epoch(model, val_loader, DEVICE)

        if scheduler is not None:
            scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr_now,
            "time_sec": elapsed,
        }
        history.append(record)

        last_train_loss = train_loss
        last_val_loss = val_loss

        # Save last checkpoint every epoch
        save_last_checkpoint(
            model, optimizer, scheduler,
            epoch, train_loss, val_loss, best_val_loss, history
        )

        # Save best checkpoint if improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_best_checkpoint(
                model, optimizer, scheduler,
                epoch, train_loss, val_loss, best_val_loss, history
            )
            print(f"[INFO] New best model saved at epoch {epoch} | val_loss={val_loss:.6f}")

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"best_val_loss={best_val_loss:.4f} | "
            f"lr={lr_now:.6f} | "
            f"time={elapsed/60:.1f} min"
        )

        # Optional persistent upload every epoch
        if ENABLE_KAGGLE_DATASET_BACKUP:
            create_or_update_kaggle_dataset(
                CKPT_DIR,
                dataset_id=KAGGLE_DATASET_ID,
                title=KAGGLE_DATASET_TITLE,
                version_note=f"epoch {epoch} | best_val_loss={best_val_loss:.6f}"
            )

        # Graceful stop after current epoch if interruption requested
        if stop_requested["flag"]:
            print("[WARN] Stop requested. Saving emergency checkpoint and exiting loop.")
            save_emergency_checkpoint(
                model, optimizer, scheduler,
                epoch, train_loss, val_loss, best_val_loss, history
            )
            if ENABLE_KAGGLE_DATASET_BACKUP:
                create_or_update_kaggle_dataset(
                    CKPT_DIR,
                    dataset_id=KAGGLE_DATASET_ID,
                    title=KAGGLE_DATASET_TITLE,
                    version_note=f"emergency stop at epoch {epoch}"
                )
            break

except Exception as e:
    print(f"[ERROR] Training interrupted by exception: {e}")
    save_emergency_checkpoint(
        model, optimizer, scheduler,
        last_epoch_seen,
        -1.0 if last_train_loss is None else last_train_loss,
        -1.0 if last_val_loss is None else last_val_loss,
        best_val_loss,
        history
    )
    if ENABLE_KAGGLE_DATASET_BACKUP:
        create_or_update_kaggle_dataset(
            CKPT_DIR,
            dataset_id=KAGGLE_DATASET_ID,
            title=KAGGLE_DATASET_TITLE,
            version_note=f"exception backup at epoch {last_epoch_seen}"
        )
    raise

finally:
    print("\n[INFO] Checkpoint summary")
    print("Best model      :", BEST_MODEL_PATH, "| exists =", BEST_MODEL_PATH.exists())
    print("Last checkpoint :", LAST_CKPT_PATH, "| exists =", LAST_CKPT_PATH.exists())
    print("Emergency ckpt  :", EMERGENCY_CKPT_PATH, "| exists =", EMERGENCY_CKPT_PATH.exists())
    print("History CSV     :", HISTORY_CSV_PATH, "| exists =", HISTORY_CSV_PATH.exists())

In [ ]:
# ============================================================
# 9) Training / validation loops
# ============================================================
from tqdm.auto import tqdm
import numpy as np
import time

def get_total_loss(output):
    if isinstance(output, dict):
        if "loss" in output:
            return output["loss"]
        if "total_loss" in output:
            return output["total_loss"]
    if torch.is_tensor(output):
        return output
    if isinstance(output, (list, tuple)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 0:
                return item
    raise RuntimeError(f"Could not extract loss from output type: {type(output)}")

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader))
    for images, targets in pbar:
        images = images.to(device)
        targets = {k: v.to(device) if torch.is_tensor(v) else v for k, v in targets.items()}

        optimizer.zero_grad()
        output = model(images, targets)
        loss = get_total_loss(output)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_description(f"train_loss={loss.item():.4f}")

    return running_loss / max(1, len(loader))

@torch.no_grad()
def valid_one_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader))
    for images, targets in pbar:
        images = images.to(device)
        targets = {k: v.to(device) if torch.is_tensor(v) else v for k, v in targets.items()}

        output = model(images, targets)
        loss = get_total_loss(output)

        running_loss += loss.item()
        pbar.set_description(f"val_loss={loss.item():.4f}")

    return running_loss / max(1, len(loader))


history = []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = valid_one_epoch(model, val_loader, DEVICE)

    scheduler.step()

    epoch_time = time.time() - start
    lr_now = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": lr_now,
        "time_sec": epoch_time,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"lr={lr_now:.6f} | "
        f"time={epoch_time/60:.1f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "history": history,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE,
                "num_classes": NUM_CLASSES,
                "label_to_name": label_to_name,
            },
            BEST_MODEL_PATH,
        )
        print(f"Saved best model to: {BEST_MODEL_PATH}")

print("Training finished.")
print("Best val loss:", best_val_loss)

In [ ]:
# ============================================================
# 10) Training curves
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
display(hist_df)

plt.figure(figsize=(8, 5))
plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train_loss")
plt.plot(hist_df["epoch"], hist_df["val_loss"], marker="o", label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{MODEL_NAME} on ENA24")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# ============================================================
# 11) Load best checkpoint for inference
# ============================================================
from effdet import create_model

predict_model = create_model(
    MODEL_NAME,
    bench_task="predict",
    num_classes=NUM_CLASSES,
    pretrained=False,
    image_size=(IMG_SIZE, IMG_SIZE),
).to(DEVICE)

ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
predict_model.load_state_dict(ckpt["model_state_dict"])
predict_model.eval()

print("Loaded best checkpoint from:", BEST_MODEL_PATH)

In [ ]:
# ============================================================
# 12) Visualization of predictions
#    Output format expected: [x1, y1, x2, y2, score, class_id]
# ============================================================
import matplotlib.patches as patches
import numpy as np

@torch.no_grad()
def show_predictions(model, dataset, device, num_images=4, score_threshold=0.30):
    indices = list(range(min(num_images, len(dataset))))

    for idx in indices:
        image, target = dataset[idx]
        inp = image.unsqueeze(0).to(device)

        preds = model(inp)[0].detach().cpu().numpy()

        img_np = image.permute(1, 2, 0).numpy()

        fig, ax = plt.subplots(1, figsize=(9, 9))
        ax.imshow(img_np)

        kept = 0
        for det in preds:
            x1, y1, x2, y2, score, cls_id = det.tolist()
            if score < score_threshold:
                continue
            cls_id = int(cls_id)
            if cls_id <= 0:
                continue

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2,
                edgecolor="red",
                facecolor="none"
            )
            ax.add_patch(rect)

            class_name = label_to_name.get(cls_id, f"class_{cls_id}")
            ax.text(
                x1,
                max(0, y1 - 5),
                f"{class_name} {score:.2f}",
                color="white",
                fontsize=10,
                bbox=dict(facecolor="red", alpha=0.6, edgecolor="none"),
            )
            kept += 1

        ax.set_title(f"Predictions kept: {kept}")
        ax.axis("off")
        plt.show()

show_predictions(predict_model, val_ds, DEVICE, num_images=4, score_threshold=0.30)

In [ ]:
!pip install -q pycocotools

In [ ]:
from pycocotools.coco import COCO

coco_gt = COCO(str(VAL_JSON))
gt_ids = set(coco_gt.getImgIds())

print("Sample GT ids:", list(gt_ids)[:10])

images, targets = next(iter(val_loader))
pred_ids = targets["image_id"]

print("Sample loader ids:", pred_ids[:10])
print("Type first loader id:", type(pred_ids[0]))

bad_ids = [x for x in pred_ids if x not in gt_ids]
print("Bad ids in first batch:", bad_ids[:10])

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm
import torch

label_to_cat_id = {v: k for k, v in cat_id_to_label.items()}

@torch.no_grad()
def evaluate_coco(model, dataloader, device, ann_json, score_threshold=0.05):
    model.eval()

    coco_gt = COCO(str(ann_json))
    valid_img_ids = set(coco_gt.getImgIds())
    results = []

    for images, targets in tqdm(dataloader, total=len(dataloader), desc="Evaluating"):
        images = images.to(device)
        preds = model(images)

        for i in range(len(images)):
            image_id = targets["image_id"][i]

            # normalize image_id type to match COCO ground truth ids
            try:
                if image_id not in valid_img_ids:
                    image_id = int(image_id)
            except:
                pass

            if image_id not in valid_img_ids:
                continue

            dets = preds[i].detach().cpu().numpy()

            for det in dets:
                x1, y1, x2, y2, score, cls = det.tolist()

                if score < score_threshold:
                    continue

                cls = int(cls)
                if cls <= 0:
                    continue
                if cls not in label_to_cat_id:
                    continue

                w = x2 - x1
                h = y2 - y1

                results.append({
                    "image_id": image_id,
                    "category_id": label_to_cat_id[cls],
                    "bbox": [float(x1), float(y1), float(w), float(h)],
                    "score": float(score),
                })

    if len(results) == 0:
        print("No valid detections.")
        return None

    # final safety filter
    results = [r for r in results if r["image_id"] in valid_img_ids]

    print("Number of predictions kept:", len(results))
    print("Sample prediction image_ids:", [r["image_id"] for r in results[:10]])

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return coco_eval

In [ ]:
from effdet import DetBenchPredict

predict_model = DetBenchPredict(model.model)
predict_model = predict_model.to(DEVICE)
predict_model.eval()

coco_eval = evaluate_coco(
    predict_model,
    val_loader,
    DEVICE,
    ann_json=VAL_JSON,
    score_threshold=0.05
)